# 00 — Conversão de Dados: EDF → Espectrogramas STFT

Lê os arquivos `.edf` e anotações de crises da base **Siena Scalp EEG**, extrai janelas de 90 s para as 4 classes (`pre_ictal`, `ictal`, `pos_ictal`, `normal`), gera os espectrogramas STFT em `data/processed/spectrograms/` e salva o manifesto `data/annotations/annotations.csv`.

In [ ]:
import gc
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    load_edf,
    parse_seizure_list,
    extract_segments,
    segment_to_spectrogram,
    save_spectrogram,
)
from src.preprocessing.stft_converter import plot_class_grid

%matplotlib inline
import matplotlib.pyplot as plt

DATA_ROOT  = PROJECT_ROOT / 'physionet.org' / 'files' / 'siena-scalp-eeg' / '1.0.0'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'spectrograms'
ANNOT_PATH = PROJECT_ROOT / 'data' / 'annotations' / 'annotations.csv'

WINDOW_S = 90
NPERSEG  = 256
NOVERLAP = 128
PATIENTS = ['PN00', 'PN01', 'PN03', 'PN05', 'PN06', 'PN07', 'PN09', 'PN11', 'PN12', 'PN13', 'PN14', 'PN16', 'PN17']
CLASSES  = ['pre_ictal', 'ictal', 'pos_ictal', 'normal']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
records = []

for pid in tqdm(PATIENTS, desc='Pacientes'):
    patient_dir = DATA_ROOT / pid
    seizure_txt = patient_dir / f'Seizures-list-{pid}.txt'
    if not patient_dir.exists() or not seizure_txt.exists():
        continue

    seizures = parse_seizure_list(str(seizure_txt))
    out_pid_dir = OUTPUT_DIR / pid
    out_pid_dir.mkdir(parents=True, exist_ok=True)

    edf_to_seizures = defaultdict(list)
    for sz in seizures:
        edf_to_seizures[sz['edf_file']].append(sz)

    for edf_name, sz_list in edf_to_seizures.items():
        edf_path = patient_dir / edf_name
        if not edf_path.exists():
            continue

        raw, meta = load_edf(str(edf_path))
        all_starts = [s['sz_start_rel_s'] for s in sz_list]
        all_ends   = [s['sz_end_rel_s'] for s in sz_list]

        for sz in sz_list:
            seg_dict = extract_segments(
                raw=raw,
                sz_start_rel_s=sz['sz_start_rel_s'],
                sz_end_rel_s=sz['sz_end_rel_s'],
                all_sz_starts=all_starts,
                all_sz_ends=all_ends,
                window_s=WINDOW_S,
                fs=meta['fs'],
            )

            for cls, segment in seg_dict.items():
                out_name = f"{edf_name.replace('.edf', '')}_seizure{sz['seizure_n']:02d}_{cls}.png"
                out_path = out_pid_dir / out_name

                if segment is None:
                    status, sp_path = 'skipped_no_data', None
                else:
                    img = segment_to_spectrogram(segment, fs=meta['fs'], nperseg=NPERSEG, noverlap=NOVERLAP)
                    save_spectrogram(img, str(out_path))
                    status, sp_path = 'ok', str(out_path.relative_to(PROJECT_ROOT))

                records.append({
                    'patient_id': pid,
                    'edf_file': edf_name,
                    'seizure_n': sz['seizure_n'],
                    'sz_start_rel_s': sz['sz_start_rel_s'],
                    'sz_end_rel_s': sz['sz_end_rel_s'],
                    'sz_duration_s': sz['sz_duration_s'],
                    'class': cls,
                    'spectrogram_path': sp_path,
                    'status': status,
                })
            del seg_dict
        del raw
        gc.collect()

df_annot = pd.DataFrame(records)
df_annot.to_csv(ANNOT_PATH, index=False)
print(f'Salvo em {ANNOT_PATH} ({len(df_annot)} registros)')

In [ ]:
# Resumo e visualizacao de uma crise de exemplo
df_ok = df_annot[df_annot.status == 'ok']
display(df_ok.groupby(['patient_id', 'class']).size().unstack(fill_value=0))

if not df_ok.empty:
    sample = df_ok.iloc[0]
    spects = {}
    for cls in CLASSES:
        row = df_ok[
            (df_ok.patient_id == sample.patient_id)
            & (df_ok.edf_file == sample.edf_file)
            & (df_ok.seizure_n == sample.seizure_n)
            & (df_ok['class'] == cls)
        ]
        if not row.empty:
            spects[cls] = np.array(Image.open(PROJECT_ROOT / row.iloc[0].spectrogram_path))
    plot_class_grid(spects, sample.patient_id, sample.seizure_n, sample.edf_file)
    plt.show()